In [20]:
import torch 
from torchvision import datasets
from torch import nn, save, load
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchvision.transforms import ToTensor

In [21]:
train_data=datasets.FashionMNIST(root="./data", download=True, train=True, transform=ToTensor())
test_data=datasets.FashionMNIST(root="./data",  download= True,train =False, transform = ToTensor())

In [53]:
train= DataLoader(train_data, batch_size=32, shuffle=True)
test= DataLoader(test_data, batch_size=32, shuffle= False)

In [23]:
train

In [24]:
train_data

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [25]:
test_data

Dataset FashionMNIST
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [36]:
class MLP (nn.Module):
    def __init__(self):
        super().__init__()
        self.model= nn.Sequential(
            nn.Flatten(),
            nn.Linear(784,256),
            nn.Linear(256,128),
            nn.Linear(128,64),   
            nn.Linear(64,10),   
        )
    def forward(self,x):
        return self.model(x)

In [37]:
torch.backends.mps.is_available()

True

In [38]:
device= torch.device('mps')

In [39]:
device

device(type='mps')

In [40]:
model=MLP()

In [41]:
model.to(device)

MLP(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [42]:
loss=nn.CrossEntropyLoss()

In [43]:
optim= AdamW(model.parameters(),lr=1e-3)

In [44]:
epochs=5

In [47]:
for epoch in tqdm(range(epochs)):
    epoch_loss=0.0
    for batch in train:
        x,y= batch
        x=x.to(device)
        y=y.to(device)
        y_hat=model(x)
        L=loss(y_hat,y)
        optim.zero_grad()
        L.backward()
        optim.step()
        epoch_loss+=L.item()
    print(f"Loss for epoch {epoch+1} : {epoch_loss/len(train)}/")

 20%|██        | 1/5 [00:01<00:04,  1.23s/it]

Loss for epoch 1 : 0.4475491828621386/


 40%|████      | 2/5 [00:02<00:03,  1.22s/it]

Loss for epoch 2 : 0.4390986403242087/


 60%|██████    | 3/5 [00:03<00:02,  1.22s/it]

Loss for epoch 3 : 0.43168222775664955/


 80%|████████  | 4/5 [00:04<00:01,  1.22s/it]

Loss for epoch 4 : 0.42569336228477306/


100%|██████████| 5/5 [00:06<00:00,  1.22s/it]

Loss for epoch 5 : 0.41944904554004486/


In [48]:
with open('FashionMNIST_mlp.pt','wb') as f:
    save(model,f)

In [49]:
mlp = load('FashionMNIST_mlp.pt',weights_only=False)

In [50]:
mlp

MLP(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [51]:
mlp.eval()

MLP(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [52]:
mlp.to(device)

MLP(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [56]:
with torch.no_grad():
    total=0
    correct=0
    for batch in test:
        x,y= batch
        x=x.to(device)
        y=y.to(device)
        y_hat=model(x)
        preds= y_hat.argmax(dim=1)
        correct+=(preds==y).sum().item()
        total+=y.size(0)
    print(f"accuracy = {correct/total *100}")
    


accuracy = 85.79
